# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emaanzehra/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row equals one content page, identified by content_id. The time window is a single mid-panel month: 2026-03 (March 2026). Each row summarises that page's search performance signals aggregated over the 90 days prior to the decision point.

In [9]:
import os
os.environ["HF_TOKEN"] = __import__("google.colab", fromlist=["userdata"]).userdata.get("HF_TOKEN")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
result = con.sql(f"""
SELECT COUNT(*) as row_count,
       COUNT(DISTINCT content_hash_id) as unique_pages,
       MIN(report_date) as earliest,
       MAX(report_date) as latest
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(result)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   row_count  unique_pages   earliest     latest
0    9841378        331437 2026-03-01 2026-03-31


## 2. Fields: feature / label / context / excluded

Features: impressions_90d, avg_position, ctr, days_since_last_update, content_age_days. Label: is_declining_label, defined as trend_direction == "down". Context: client_hash_id, content_hash_id. Excluded: trend_pct, because it is derived directly from the label and would cause leakage.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
import pandas as pd
import numpy as np

features = con.sql(f"""
SELECT content_hash_id,
       AVG(gsc_impressions) as avg_impressions,
       AVG(gsc_avg_position) as avg_position,
       AVG(gsc_clicks) as avg_clicks,
       COUNT(DISTINCT report_date) as days_with_data,
       MAX(gsc_impressions) as max_impressions,
       -- LEAKY COLUMN (will be removed):
       AVG(gsc_clicks) / NULLIF(AVG(gsc_impressions), 0) as leaky_ctr_ratio
FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
WHERE (gsc_impressions IS NOT NULL) IS TRUE
GROUP BY content_hash_id
LIMIT 5000
""").df()

print("Feature frame shape:", features.shape)
print(features.head())

print("\nWith leaky column - correlation to avg_clicks:",
      features["leaky_ctr_ratio"].corr(features["avg_clicks"]))

features = features.drop(columns=["leaky_ctr_ratio"])
print("Leaky column removed. Final features:", list(features.columns))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (5000, 7)
            content_hash_id  avg_impressions  avg_position  avg_clicks  \
0  content_d0dff76c889de68f         5.838710      5.147402    0.000000   
1  content_67741cce996cfafa         1.483871      4.828125    0.032258   
2  content_2e6360ad20fd7107        29.000000      5.145765    0.032258   
3  content_ac8663da7484669a         1.096774      4.909314    0.000000   
4  content_65c50dfe9d87a585       100.258065      6.969536    0.000000   

   days_with_data  max_impressions  leaky_ctr_ratio  
0              31               17         0.000000  
1              31                8         0.021739  
2              31              119         0.001112  
3              31                8         0.000000  
4              31              207         0.000000  

With leaky column - correlation to avg_clicks: 0.1495390855900969
Leaky column removed. Final features: ['content_hash_id', 'avg_impressions', 'avg_position', 'avg_clicks', 'days_with_data', 'max_imp

## 4. Data limits

This data slice cannot tell us whether a page declined because of a content quality issue or because of an external SERP change. The panel is unbalanced: different clients have different amounts of history, so some clients have fewer rows in March 2026 than others. Rows before a client's GA4 start date contain search data only, so engagement signals like sessions and scroll rate are missing for early rows. The March 2026 slice covers one month only, which may not represent seasonal patterns across the full year.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.